In [ ]:
import nltk
nltk.download('punkt')
nltk.download('vader_lexicon')
nltk.download("punkt_tab")


In [ ]:
import pandas as pd
from tqdm import tqdm
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk import sent_tokenize
from openai import OpenAI
from dotenv import load_dotenv
import os

# setup
sia = SentimentIntensityAnalyzer()

def vader_label(text_list):
    text = " ".join(text_list)
    sentences = sent_tokenize(text)
    scores = [sia.polarity_scores(s)['compound'] for s in sentences]
    if not scores:
        return 'neutral'
    avg = sum(scores) / len(scores)
    if avg > 0.05:
        return 'positive'
    elif avg < -0.05:
        return 'negative'
    else:
        return 'neutral'
    
# OpenAI Setup
# Current setup uses llm local LLM, but feel free to use other LLM providers like ChatGPT, Cluade, etc.
load_dotenv()
OPENAI_API_KEY = os.getenv("API_KEY")

client = OpenAI(
    api_key=OPENAI_API_KEY
)

def llm_label(text_list, model="gpt-4o-mini"):
    text = " ".join(text_list)
    prompt = f"""
You are an NLP annotator. Classify the overall sentiment of this news article toward video-based social media
(e.g., YouTube, Instagram, Snapchat) and youth mental health as one of: "positive", "neutral", or "negative".
Respond with only one word.

Article:
{text[:3000]}  # truncates long text
"""
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        output = response.choices[0].message.content.strip().lower()
        # sanitize model response
        if "pos" in output:
            return "positive"
        elif "neg" in output:
            return "negative"
        elif "neu" in output:
            return "neutral"
        return "neutral"
    except Exception as e:
        print("Error:", e)
        return "neutral"

In [12]:
# load and label dataset
file_path = './news-youtube-noagegroup-mental-2024.jsonl'
data = pd.read_json(file_path, lines=True)

vader_labels = []
llm_labels = []

for _, row in tqdm(data.iterrows(), total=len(data)):
    text = row['body']
    vader_labels.append(vader_label(text))
    llm_labels.append(llm_label(text))

data['vader_sentiment'] = vader_labels
data['llm_sentiment'] = llm_labels

100%|██████████| 1326/1326 [10:14<00:00,  2.16it/s]


In [13]:
# Save the results
output_file = "labeled_youtube_comparison.jsonl"
data[['title', 'body', 'provider', 'publication_date', 'vader_sentiment', 'llm_sentiment']].to_json(
    output_file, orient='records', lines=True
)

print(f"Saved labeled file to {output_file}")

# Quick comparison summary between Vadar and LLM sentiment categorization
print("\nVADER vs LLM label comparison:")
print(data[['vader_sentiment', 'llm_sentiment']].value_counts())
agreement = (data['vader_sentiment'] == data['llm_sentiment']).mean()
print(f"\nLabel agreement: {agreement:.2%}")


Saved labeled file to labeled_youtube_comparison.jsonl

VADER vs LLM label comparison:
vader_sentiment  llm_sentiment
negative         negative         402
positive         neutral          299
neutral          negative         210
positive         positive         141
                 negative         123
neutral          neutral          104
negative         neutral           35
neutral          positive          11
negative         positive           1
Name: count, dtype: int64

Label agreement: 48.79%
